In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
#Load data into Dataframe
df = pd.read_csv("car.data",header=None)

In [3]:
#Check the structure
df.head()

,0,1,2,3,4,5,6
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [4]:
#Add column names
df.columns = ["buying","maint","doors","persons","lug_boot","safety","class"]

In [5]:
df.shape

(1728, 7)

In [6]:
#Recheck after adding column names
df.head()

,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [7]:
#Separate features and target values
X = df.drop(columns=["class"])
y = df["class"]

In [8]:
#Turn series into numpy arrays
X = X.to_numpy()
y = y.to_numpy()

In [9]:
print(X.shape)
print(y.shape)
print(np.unique(y))

(1728, 6)
(1728,)
['acc' 'good' 'unacc' 'vgood']


In [10]:
#Entropy calculation function 
def cal_entropy(y):
    if len(y)==0:
        return 0
    classes,counts=np.unique(y,return_counts=True)
    probabilities = counts/len(y)
    entropy = -np.sum(probabilities*np.log2(probabilities))
    return entropy


In [11]:
#Create Node class
class Node:
    def __init__(self,feature=None,threshold=None,left=None,right=None,value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

#Create DecisionTree class
class DecisionTree:
    def __init__(self):
        self.root = None
    
    #fit method
    def fit(self,X,y):
        self.root = self.build_tree(X,y)
    
    #build tree method
    def build_tree(self,X,y):
        if len(np.unique(y))==1:
            classes = np.unique(y)
            node = Node()
            node.value = classes[0]
            return node
        
        best_gain,best_threshold,best_feature = self.best_split(X,y)
        
        if best_feature is None:
            classes,counts = np.unique(y,return_counts=True)
            index = np.argmax(counts)
            majority = classes[index]
            node = Node()
            node.value = majority
            return node
        
        parent_node = Node(feature = best_feature,threshold=best_threshold)
        left_split,right_split = self.split(X,y,best_feature,best_threshold)
        left_X = X[left_split]
        left_y = y[left_split]
        right_X = X[right_split]
        right_y = y[right_split]
        
        left_node = self.build_tree(left_X,left_y)
        right_node = self.build_tree(right_X,right_y)
            
        parent_node.left = left_node
        parent_node.right = right_node
        
        return parent_node
    
    #split method
    def split(self,X,y,feature,threshold):
        left_split=[]
        right_split=[]
        for i,x in enumerate(X):
            if x[feature] == threshold:
                left_split.append(i)
            else:
                right_split.append(i)
                
        return left_split,right_split
    
    #calculate information gain
    def information_gain(self,X,y,feature,threshold):
    
        left,right=self.split(X,y,feature,threshold)
        left_entropy = cal_entropy(y[left])
        right_entropy = cal_entropy(y[right])
        w_left = len(left)/len(y)
        w_right = len(right)/len(y)

        parent_entropy = cal_entropy(y)
        weighted_entropy = (left_entropy * w_left) + (right_entropy * w_right)
        info_gain = parent_entropy - weighted_entropy
        return info_gain
    
    #Choose best split
    def best_split(self,X,y):
        
        best_feature = None
        best_threshold = None
        best_gain = 0
        
        for i in range(X.shape[1]):
            thresholds = np.unique(X[:,i])
            feature = i
            for j in range(len(thresholds)):
                threshold = thresholds[j]
                
                gain = self.information_gain(X,y,feature,threshold)
                
                if gain > best_gain:
                    best_gain = gain
                    best_threshold = threshold
                    best_feature = feature
        return best_gain,best_threshold,best_feature
    
    #Predict single car
    def predict_one(self,X):
        temp = self.root
        while temp.value is None:
            if temp.threshold == X[temp.feature]:
                temp = temp.left
            else:
                temp = temp.right

        return temp.value
    
    #Predict multiple cars
    def predict(self,X):
        predictions = []
        for i in range(len(X)):
            pred = self.predict_one(X[i])
            predictions.append(pred)
        
        return np.array(predictions)
    
    

In [12]:
#Calculate accuracy of predictions
def my_accuracy(y_actual,y_predict):
    correct = 0
    for i in range(len(y_predict)):
        if y_actual[i] == y_predict[i]:
            correct+=1
    accuracy = correct/len(y_actual)
    return accuracy
        

In [13]:
#Return predictions in a confusion matrix and classes 
def my_confusion_matrix(y_actual,y_pred):
    classes = np.unique(np.concatenate((y_actual,y_pred)))
    n = len(classes)
    conf_mtrx = np.zeros((n,n),dtype=int)
    classes_to_index = {}
    for i,x in enumerate(classes):
        classes_to_index[x] = i
    
    for i in range(len(y_actual)):
        actual = y_actual[i]
        predicted = y_pred[i]
        
        actual_index = classes_to_index[actual]
        predicted_index = classes_to_index[predicted]
        
        conf_mtrx[actual_index,predicted_index] += 1
        
    return conf_mtrx,classes
        
    

In [14]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=11,stratify=y)
print("X_train shape:",X_train.shape)
print("X_test shape:",X_test.shape)
print("y_train shape:",y_train.shape)
print("y_test shape:",y_test.shape)

X_train shape: (1382, 6)
X_test shape: (346, 6)
y_train shape: (1382,)
y_test shape: (346,)


In [15]:
#Create decision tree object
tree = DecisionTree()

In [25]:
#Train the model
tree.fit(X_train,y_train)

In [26]:
#Check first ten test data prediction and compare with actual values
pred = tree.predict(X_test[:10,:])
print("Predictions for X_test:",pred)
print("Actual values:         ",y_test[:10])

Predictions for X_test: ['unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'acc' 'unacc']
Actual values:          ['unacc' 'unacc' 'unacc' 'unacc' 'acc' 'acc' 'unacc' 'unacc' 'acc' 'unacc']


In [35]:
#Predict all test data
predictions = tree.predict(X_test)

In [28]:
#Calculate accuracy using the custom function
accuracy = my_accuracy(y_test,predictions)
print("Accuracy of Decision tree classifier:",accuracy)

Accuracy of Decision tree classifier: 0.9855491329479769


In [32]:
#Verify accuracy using scikit-learn
accuracy_using_sklearn = accuracy_score(y_test, predictions)
print("Accuracy using sklearn:", accuracy_using_sklearn )

Accuracy using sklearn: 0.9855491329479769


In [36]:
#Calculate the confusion matrix using the custom function 
confusion_matrix,classes = my_confusion_matrix(y_test,predictions)

In [34]:
#Display the confusion matrix with class labels
conf_mtrx_df = pd.DataFrame(confusion_matrix,index=classes,columns=classes)
conf_mtrx_df

,acc,good,unacc,vgood
acc,72,0,4,1
good,0,14,0,0
unacc,0,0,242,0
vgood,0,0,0,13
